# Rides & Rain: Exploratory Data Analysis

DATA2201 Data Analysis, Hanbit University, Fall 2026

This notebook looks at the cleaned hourly Seoul bike-sharing data before any modelling. We want to see how demand (`rented_bikes`) changes with hour of day, rain, temperature, season and holidays.

Input: `data/clean_seoul_bike_2025.csv` (produced by the cleaning step in the README).
Output: `report/figures/demand_by_hour.png`, used in Section 4 of the report.

Note: the course subset is small (about 30 hourly rows), so every pattern below is preliminary.

In [ ]:
import os

import matplotlib.pyplot as plt
import pandas as pd

pd.set_option("display.width", 120)
plt.rcParams["figure.dpi"] = 110

DATA_PATH = "../data/clean_seoul_bike_2025.csv"
FIG_DIR = "../report/figures"
os.makedirs(FIG_DIR, exist_ok=True)

df = pd.read_csv(DATA_PATH, parse_dates=["datetime"])
print("Shape:", df.shape)
df.head()

## 1. Data check

Before looking at patterns, we confirm the cleaning rules worked: 10 columns, no duplicates, hours in 0-23, no missing values in the filled columns.

In [ ]:
df.info()
print()
print(df.describe().T.round(2))

In [ ]:
print("Season counts:\n", df["season"].value_counts(), sep="")
print()
print("Holiday counts:\n", df["holiday"].value_counts(), sep="")
print()
print("Duplicate rows:", df.duplicated().sum())
print("Hours outside 0-23:", (~df["hour"].between(0, 23)).sum())
print("Missing values per column:\n", df.isna().sum(), sep="")

## 2. Rain and demand

We flag an hour as rainy when `rain_mm > 0`. The team's finding is that rainy hours have much lower average demand than dry hours.

In [ ]:
df["is_rainy"] = df["rain_mm"] > 0
rain_summary = df.groupby("is_rainy")["rented_bikes"].agg(["count", "mean", "median"]).round(1)
rain_summary.index = ["Dry hours" if not r else "Rainy hours" for r in rain_summary.index]
print(rain_summary)

dry_mean = df.loc[~df["is_rainy"], "rented_bikes"].mean()
rainy_mean = df.loc[df["is_rainy"], "rented_bikes"].mean()
print(f"\nRainy hours average {rainy_mean:.0f} bikes vs {dry_mean:.0f} on dry hours "
      f"({1 - rainy_mean / dry_mean:.0%} lower).")

## 3. Holidays and season

The holiday label is either `Holiday` or `No Holiday` after normalisation. Season means show the largest seasonal gap in the sample.

In [ ]:
holiday_summary = df.groupby("holiday")["rented_bikes"].agg(["count", "mean"]).round(0)
print(holiday_summary)
print()

season_order = ["Winter", "Spring", "Summer", "Autumn"]
season_summary = (
    df.groupby("season")["rented_bikes"]
    .agg(["count", "mean"])
    .reindex(season_order)
    .round(0)
)
print(season_summary)

## 4. Hourly pattern (required figure)

The hourly plot compares average demand for each hour of the day on working days and on holidays. Working days should show commute peaks around 8h and 18h, while holidays should show one broad midday peak.

In [ ]:
df["day_type"] = df["holiday"].map({"No Holiday": "Working day", "Holiday": "Holiday"})

hourly = (
    df.groupby(["hour", "day_type"])["rented_bikes"]
    .mean()
    .unstack("day_type")
    .reindex(range(24))
)

fig, ax = plt.subplots(figsize=(9, 5))
for label, color in [("Working day", "tab:blue"), ("Holiday", "tab:orange")]:
    if label in hourly.columns:
        ax.plot(hourly.index, hourly[label], marker="o", label=label, color=color)

ax.set_title("Mean hourly bike rentals by hour of day, Seoul")
ax.set_xlabel("Hour of day (0-23, local time)")
ax.set_ylabel("Mean rented bikes per hour (count)")
ax.set_xticks(range(0, 24, 2))
ax.set_xlim(-0.5, 23.5)
ax.grid(alpha=0.3)
ax.legend(title="Day type")
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "demand_by_hour.png"), dpi=150)
plt.show()

print("Working-day peak hour:", hourly["Working day"].idxmax() if "Working day" in hourly else "n/a")
print("Holiday peak hour:", hourly["Holiday"].idxmax() if "Holiday" in hourly else "n/a")

## 5. Temperature and weather correlations

We check the Pearson correlation of each weather variable with demand. Temperature is the strongest positive driver in this sample; humidity and rain are weaker.

In [ ]:
weather_cols = ["temp_c", "humidity_pct", "wind_ms", "rain_mm", "snow_cm"]
corr_with_demand = (
    df[weather_cols + ["rented_bikes"]]
    .corr()["rented_bikes"]
    .drop("rented_bikes")
    .round(2)
)
print("Pearson correlation with rented_bikes:")
print(corr_with_demand.sort_values(ascending=False))

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(df["temp_c"], df["rented_bikes"], alpha=0.6, edgecolor="k", linewidth=0.3)
ax.set_title("Air temperature vs hourly rentals")
ax.set_xlabel("Air temperature (°C)")
ax.set_ylabel("Rented bikes per hour (count)")
ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()

## 6. Summary of EDA findings

- Rainy hours average about 214 rented bikes against about 812 on dry hours, a reduction of roughly 74%.
- Temperature has a Pearson correlation of about 0.54 with rentals, the strongest weather link in this sample.
- Holidays average about 562 rentals per hour against about 733 on non-holidays.
- Summer averages about 1,020 rentals per hour and Winter about 305.
- Working days show commute peaks at 8h and 18h; holidays show a single broad peak around 14-16h.

These results guide the feature choices for modelling: cyclic hour encoding, a rain flag, temperature and humidity, and season. Because the subset is small and the split is random, we treat these as descriptive patterns, not firm estimates.